# 04 · Posterior reconstruction + uncertainty

Run NUTS over `z`, decode each posterior sample, and form the per-pixel
**mean** and **std**. The std map is the uncertainty. Keep `latent_dim` ≈
128–256 so the sampler mixes.

In [ ]:
import jax.numpy as jnp, matplotlib.pyplot as plt
from mrigen.train_vae import load_model
from mrigen.data import FastMRISlices
from mrigen.fourier import fft2c
from mrigen.masks import equispaced_mask
from mrigen.recon.vae_numpyro import reconstruct_posterior
from mrigen import viz, metrics

In [ ]:
vae = load_model('../checkpoints/vae_128.eqx', latent_dim=128)
ds = FastMRISlices('../data/processed')
x = jnp.asarray(ds[0])
mask = equispaced_mask((128, 128), acceleration=4)
y = mask * fft2c(x)

In [ ]:
post = reconstruct_posterior(y, mask, vae.decoder, vae.latent_dim,
                             sigma=0.01, num_warmup=200, num_samples=200)
viz.panel(x, post['mean'], std=post['std']); plt.show()
print('diversity', metrics.diversity(post['samples']))

### Calibration: does predicted std track actual error?

In [ ]:
import numpy as np
err = np.abs(np.asarray(post['mean']) - np.asarray(x))
s, e = metrics.calibration_curve(err, np.asarray(post['std']))
plt.plot(s, e, 'o-'); plt.xlabel('predicted std'); plt.ylabel('mean |error|'); plt.show()